# Movie Recommendation and Rating Analysis Using Machine Learning

This notebook walks through the full workflow for the project:

1. Load and clean the `movies.csv` dataset
2. Exploratory Data Analysis (EDA) on ratings, popularity, and release years
3. Build a **content-based recommendation system** using TF-IDF on movie overviews + cosine similarity
4. Evaluate the recommender with example queries
5. Save the trained model artifacts for use by `app.py` / `recommendation.py`


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("../data/movies.csv")
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
print(df.shape)
df.head()


## 2. Data Cleaning

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate titles:", df["title"].duplicated().sum())
print("Duplicate ids:", df["id"].duplicated().sum())


In [ ]:
df["overview"] = df["overview"].fillna("")
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

for col in ["popularity", "vote_average", "vote_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["id", "title"]).drop_duplicates(subset="id", keep="first")
df = df.reset_index(drop=True)

print(df.shape)
df.describe()


## 3. Exploratory Data Analysis

### 3.1 Rating distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["vote_average"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Vote Average")
axes[0].set_xlabel("Vote Average")

sns.histplot(np.log1p(df["popularity"]), bins=30, kde=True, ax=axes[1], color="orange")
axes[1].set_title("Distribution of log(1 + Popularity)")
axes[1].set_xlabel("log(1 + Popularity)")

plt.tight_layout()
plt.show()


### 3.2 Popularity vs. rating

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="popularity", y="vote_average", alpha=0.3)
plt.xscale("log")
plt.title("Popularity vs. Vote Average")
plt.xlabel("Popularity (log scale)")
plt.ylabel("Vote Average")
plt.show()

print("Correlation (popularity vs vote_average):", df["popularity"].corr(df["vote_average"]).round(3))


### 3.3 Movies released per year

In [ ]:
by_year = df.dropna(subset=["release_year"]).groupby("release_year").size()
by_year = by_year[by_year.index >= 1960]

plt.figure(figsize=(12, 5))
by_year.plot(kind="bar", width=0.9)
plt.title("Movies per Release Year (1960+)")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.xticks(rotation=90)
plt.show()


### 3.4 Top rated & most popular movies

In [ ]:
top_rated = df[df["vote_count"] >= 1000].sort_values("vote_average", ascending=False).head(10)
top_rated[["title", "release_year", "vote_average", "vote_count"]]


In [ ]:
most_popular = df.sort_values("popularity", ascending=False).head(10)
most_popular[["title", "release_year", "popularity", "vote_average"]]


## 4. Building the Content-Based Recommendation Model

We use **TF-IDF (Term Frequency – Inverse Document Frequency)** on each movie's `overview`
text, then rank candidate movies by **cosine similarity** between TF-IDF vectors.
This is a classic content-based filtering approach: movies that "talk about similar
things" in their plot summaries end up close together in TF-IDF space.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=25000,
    ngram_range=(1, 2),
    min_df=2,
)

tfidf_matrix = vectorizer.fit_transform(df["overview"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


### 4.1 Title lookup index

Titles are not unique in this dataset (remakes, shared names), so we index by row
position and, for duplicate titles, prefer the most popular entry.

In [ ]:
title_index = {}
order = df.sort_values("popularity", ascending=False).index
for pos in order:
    key = str(df.loc[pos, "title"]).strip().lower()
    title_index.setdefault(key, []).append(pos)

print("Unique title keys:", len(title_index))


### 4.2 Recommendation function

In [ ]:
def recommend(title, top_n=10):
    key = title.strip().lower()
    if key not in title_index:
        raise ValueError(f"'{title}' not found in dataset")
    idx = title_index[key][0]

    sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    order = sims.argsort()[::-1]
    order = [i for i in order if i != idx][:top_n]

    result = df.iloc[order][["title", "release_year", "vote_average"]].copy()
    result["similarity"] = sims[order]
    return result.reset_index(drop=True)

recommend("The Godfather", top_n=10)


### 4.3 More example queries

In [ ]:
recommend("The Shawshank Redemption", top_n=10)


In [ ]:
recommend("Inception", top_n=10) if "inception" in title_index else print("Title not in this dataset sample")


## 5. Discussion

- **Approach**: content-based filtering on plot overviews is a good fit here because
  the dataset has no user-level rating history (no user/item ratings matrix), only
  per-movie metadata — so a collaborative-filtering approach isn't possible without
  additional data. TF-IDF + cosine similarity is fast, interpretable, and needs no
  training labels.
- **Limitations**: recommendations rely entirely on how similar plot descriptions are
  in wording; two movies with the same theme but very differently written overviews
  may not be judged similar. Popularity/genre metadata isn't used here (the dataset
  doesn't include genres), which would be a natural extension.
- **Possible extensions**: combine overview similarity with a rating/popularity bonus
  (hybrid scoring), add genre or cast data if available, or swap TF-IDF for sentence
  embeddings for more semantic matching.

## 6. Save Model Artifacts

This mirrors what `train.py` does, so the trained model can be reused by `app.py`.

In [ ]:
import pickle
from pathlib import Path

model_path = Path("../models/movie_similarity.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

with open(model_path, "wb") as f:
    pickle.dump(
        {
            "vectorizer": vectorizer,
            "tfidf_matrix": tfidf_matrix,
            "df": df,
            "title_index": title_index,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("Saved to", model_path.resolve())
